# Binary Decomposition – Experiment Dashboard

Postup experimentov:
1. **Prehľad datasetov** – vizualizácia a štatistiky vstupných dát
2. **Analýza hyperparametrov** (`analysis/objects_unique`, `analysis/leafs_subset`) – hľadanie optimálnych parametrov
   - EXP-1: pop_size a patience
   - EXP-2: Inicializačná metóda GA
   - EXP-3: Metóda crossoveru
   - EXP-5: Mutačné pravdepodobnosti (p_local, p_merge)
   - Konvergečná analýza
3. **Full Run** (`leafs_selected`, `objects_selected`) – finálne porovnanie algoritmov

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('../../..')  # from notebooks/dashboard/
CSV_DIR  = PROJECT_ROOT / 'experiments/results/csv'
DATA_DIR = PROJECT_ROOT / 'data/datasets'

# ── Publication-ready style ──────────────────────────────────────────
plt.rcParams.update({
    # Background
    'figure.facecolor':      'white',
    'axes.facecolor':        'white',
    'savefig.facecolor':     'white',
    'savefig.edgecolor':     'white',
    # Text colors — explicit dark so nothing disappears on white
    'text.color':            '#222222',
    'axes.labelcolor':       '#222222',
    'axes.titlecolor':       '#222222',
    'xtick.color':           '#444444',
    'ytick.color':           '#444444',
    'xtick.labelcolor':      '#222222',
    'ytick.labelcolor':      '#222222',
    'legend.labelcolor':     '#222222',
    # Grid
    'axes.grid':             True,
    'grid.color':            '#e0e0e0',
    'grid.linewidth':        0.8,
    'axes.axisbelow':        True,
    # Spines
    'axes.spines.top':       False,
    'axes.spines.right':     False,
    'axes.spines.left':      True,
    'axes.spines.bottom':    True,
    'axes.edgecolor':        '#444444',
    'axes.linewidth':        0.8,
    # Font
    'font.family':           'sans-serif',
    'font.size':             11,
    'axes.titlesize':        13,
    'axes.labelsize':        11,
    'legend.fontsize':       10,
    'legend.framealpha':     0.9,
    'legend.edgecolor':      '#cccccc',
    'legend.facecolor':      'white',
    # Lines
    'lines.linewidth':       2.0,
    'lines.antialiased':     True,
    # Output quality
    'figure.dpi':            150,
    'savefig.dpi':           300,
    'savefig.bbox':          'tight',
    'savefig.pad_inches':    0.1,
})

In [ ]:
# ── Helper: load all results CSVs for a given algorithm ──────────────
def load_results(algorithm: str, dataset: str, run_id: str = None) -> pd.DataFrame:
    """Load results CSV for an algorithm/dataset combination.
    
    If run_id is None, loads and concatenates all run subdirectories.
    """
    base = CSV_DIR / algorithm
    if not base.exists():
        return pd.DataFrame()
    
    dfs = []
    pattern = f"{dataset}/{run_id}/results.csv" if run_id else f"{dataset}/**/results.csv"
    for csv_path in sorted(base.glob(pattern)):
        try:
            df = pd.read_csv(csv_path)
            df['algorithm'] = algorithm
            df['dataset'] = dataset
            df['run_path'] = str(csv_path.parent.relative_to(base))
            dfs.append(df)
        except Exception:
            pass
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def load_generations(algorithm: str, dataset: str) -> pd.DataFrame:
    """Load generation history CSVs for a GA algorithm."""
    base = CSV_DIR / algorithm
    if not base.exists():
        return pd.DataFrame()
    dfs = []
    for csv_path in sorted(base.glob(f"{dataset}/**/generations.csv")):
        try:
            df = pd.read_csv(csv_path)
            df['algorithm'] = algorithm
            df['run_path'] = str(csv_path.parent.relative_to(base))
            dfs.append(df)
        except Exception:
            pass
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print('Helpers loaded.')

In [ ]:
DETERMINISTIC_ALGOS = ['dm', 'gdm', 'quadtree', 'largest_rect', 'graph_based']
GA_ALGOS = ['ga_dm', 'ga_gdm', 'ga_random', 'ga_qtd', 'ga_lrf']
DATASETS_COMPARE = ['leafs_selected', 'objects_selected']

def load_baseline(algo: str, ds: str) -> pd.DataFrame:
    """Load baseline run — any run_id containing 'run1'."""
    df = load_results(algo, ds)
    if df.empty:
        return df
    mask = (
        ~df['run_path'].str.contains('test|exp2|exp3', na=False)
        & df['run_path'].str.contains('run1', na=False)
    )
    return df[mask].drop_duplicates(subset='image_name')

def load_ga_baseline(algo: str, ds: str) -> pd.DataFrame:
    """Load GA results — non-test/exp, deduplicated per image."""
    df = load_results(algo, ds)
    if df.empty:
        return df
    df = df[~df['run_path'].str.contains('test|exp2|exp3', na=False)]
    return df.sort_values('rectangle_count').drop_duplicates(subset='image_name')

print('Baseline helpers loaded.')

## 1. Prehľad datasetov

In [ ]:
# Load manifests if available, otherwise scan npy dirs
def dataset_info(name: str) -> pd.DataFrame:
    manifest = DATA_DIR / name / 'manifest.csv'
    if manifest.exists():
        return pd.read_csv(manifest)
    # Fall back: scan npy dir for shapes
    npy_dir = DATA_DIR / name / 'npy'
    rows = []
    for p in sorted(npy_dir.glob('*.npy')):
        arr = np.load(p, mmap_mode='r')
        h, w = arr.shape[0], arr.shape[1]
        rows.append({'image_name': p.name, 'height': h, 'width': w, 'pixels': h * w})
    return pd.DataFrame(rows)

leafs_info = dataset_info('leafs_selected')
objects_info = dataset_info('objects_selected') if (DATA_DIR / 'objects_selected').exists() else pd.DataFrame()

print(f"Leafs:   {len(leafs_info)} images, "
      f"pixels {leafs_info['pixels'].min():,} – {leafs_info['pixels'].max():,}")
if not objects_info.empty:
    print(f"Objects: {len(objects_info)} images, "
          f"pixels {objects_info['pixels'].min():,} – {objects_info['pixels'].max():,}")

In [ ]:
for info, title, fname in [
    (leafs_info,   'Listy (leafs_selected)',     'fig_01a_dataset_leafs.png'),
    (objects_info, 'Objekty (objects_selected)', 'fig_01b_dataset_objects.png'),
]:
    if info.empty:
        print(f'No data for {title}')
        continue
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(info['pixels'] / 1000, bins=30, color='steelblue', edgecolor='white')
    ax.set_xlabel('Veľkosť obrázka (kpx)')
    ax.set_ylabel('Počet obrázkov')
    ax.set_title(f'{title}\n(n={len(info)})')
    plt.tight_layout()
    plt.savefig(fname, bbox_inches='tight')
    plt.show()

## 2. Analýza hyperparametrov

Všetky experimenty tejto sekcie sú spúšťané na **analysis datasetoch**
(`objects_unique`, `leafs_subset`, max 10 obrázkov).
Výsledky určujú finálnu konfiguráciu pre full run.

### EXP-1A – Vplyv pop_size (ga_gdm, analysis datasety)

In [ ]:
pop_records = []
for algo in ['ga_gdm', 'ga_dm']:
    for ds in ['analysis/objects_unique', 'analysis/leafs_subset']:
        df = load_results(algo, ds)
        if df.empty:
            continue
        exp1 = df[df['run_path'].str.contains('exp1_popsize|exp1_patience', na=False)].copy()
        if exp1.empty:
            continue
        exp1['algo'] = algo
        exp1['sweep'] = exp1['run_path'].str.extract(r'(exp1_\w+)')
        exp1['value'] = exp1['run_path'].str.extract(r'exp1_\w+/(\d+)').astype(float)
        for _, row in exp1.iterrows():
            pop_records.append({
                'algo': algo,
                'sweep': row['sweep'],
                'value': row['value'],
                'dataset': ds,
                'image_name': row['image_name'],
                'rectangle_count': row['rectangle_count'],
                'execution_time_sec': row['execution_time_sec'],
                'generations_used': row.get('generations_used'),
            })

pop_df = pd.DataFrame(pop_records)
if not pop_df.empty:
    for sweep in ['exp1_popsize', 'exp1_patience']:
        sub = pop_df[pop_df['sweep'] == sweep]
        if sub.empty:
            continue
        print(f'\n=== {sweep} ===')
        display(sub.groupby(['algo', 'value', 'dataset'])
                [['rectangle_count', 'execution_time_sec']]
                .agg(['mean', 'std']).round(2))
else:
    print('EXP-1 results not available yet.')
    print('Run: python -m experiments.scripts.analysis.run_ga_exp1_popsize')
    print('     python -m experiments.scripts.analysis.run_ga_exp1_dm_popsize')

In [ ]:
pop_size_df = pop_df[pop_df['sweep'] == 'exp1_popsize'] if not pop_df.empty else pd.DataFrame()

if not pop_size_df.empty:
    algos = pop_size_df['algo'].unique()
    ds_list = pop_size_df['dataset'].unique()
    fig, axes = plt.subplots(len(algos), len(ds_list),
                             figsize=(6 * len(ds_list), 5 * len(algos)),
                             squeeze=False)

    for row_i, algo in enumerate(algos):
        for col_i, ds in enumerate(ds_list):
            ax = axes[row_i][col_i]
            sub = pop_size_df[(pop_size_df['algo'] == algo) & (pop_size_df['dataset'] == ds)]
            if sub.empty:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
                ax.set_title(f'{algo} | {ds.replace("analysis/", "")}')
                continue
            stats = sub.groupby('value')['rectangle_count'].agg(['mean', 'std']).reset_index()
            ax.plot(stats['value'], stats['mean'], 'o-',
                    color='steelblue', linewidth=2, markersize=7)
            ax.fill_between(stats['value'],
                            stats['mean'] - stats['std'],
                            stats['mean'] + stats['std'],
                            alpha=0.15, color='steelblue')
            ax.set_xlabel('pop_size')
            ax.set_ylabel('Mean rectangle count')
            ax.set_title(f'{algo} | {ds.replace("analysis/", "")}')
            ax.set_xticks(stats['value'].tolist())

    plt.suptitle('EXP-1A: Vplyv pop_size na počet obdĺžnikov', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig('fig_exp1a_popsize.png', bbox_inches='tight')
    plt.show()
else:
    print('EXP-1A results not available yet.')
    print('Run: python -m experiments.scripts.analysis.run_ga_exp1_popsize')
    print('     python -m experiments.scripts.analysis.run_ga_exp1_dm_popsize')

### EXP-1B – Analýza patience

Hľadáme minimálnu hodnotu patience kde GA stále konverguje.
Analýza z `generations.csv` dát — proxy cez `best_rectangle_count`.

In [ ]:
def last_improvement_gen(gen_df: pd.DataFrame):
    """For each image, find the last generation where fitness improved.

    Falls back to best_rectangle_count if fitness column is missing/all-NaN.
    """
    use_fitness = (
        'fitness' in gen_df.columns
        and gen_df['fitness'].notna().any()
    )
    col = 'fitness' if use_fitness else 'best_rectangle_count'
    # fitness: higher = better; rect_count: lower = better
    improved = lambda prev, curr: curr > prev if use_fitness else curr < prev
    print(f"Analýza patience podľa: '{col}' "
          f"({'fitness' if use_fitness else 'rect_count — proxy, menej presné'})")

    results = []
    for img_name, grp in gen_df.groupby('image_name'):
        grp = grp.sort_values('generation')
        vals = grp[col].values
        gens = grp['generation'].values
        best = vals[0]
        last_imp = gens[0]
        for g, v in zip(gens[1:], vals[1:]):
            if pd.notna(v) and improved(best, v):
                best = v
                last_imp = g
        results.append({
            'image_name': img_name,
            'last_improvement_gen': last_imp,
            'total_gens': gens[-1],
        })
    return pd.DataFrame(results)


patience_dfs = []
for algo in ['ga_gdm', 'ga_dm', 'ga_random', 'ga_qtd', 'ga_lrf']:
    for ds in DATASETS_COMPARE:
        gdf = load_generations(algo, ds)
        if gdf.empty:
            continue
        gdf['algo'] = algo
        gdf['ds'] = ds
        patience_dfs.append(gdf)

if patience_dfs:
    all_gens = pd.concat(patience_dfs, ignore_index=True)
    limp = last_improvement_gen(all_gens)
    print(f"Obrazky: {len(limp)}\n")
    for p in [50, 75, 90, 95, 99]:
        val = limp['last_improvement_gen'].quantile(p / 100)
        print(f"  p{p:2d}: generácia {val:.0f}  "
              f"→ patience={val:.0f} stačí pre {p}% obrazkov")
else:
    print('Žiadne generations.csv dáta nenájdené.')
    print('Spusti aspoň jeden GA experiment.')

In [ ]:
if patience_dfs:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Histogram last_improvement_gen
    ax = axes[0]
    ax.hist(limp['last_improvement_gen'], bins=30,
            color='steelblue', edgecolor='white')
    for p, color, ls in [(50, 'orange', '--'), (90, 'tomato', '--'),
                         (95, 'crimson', ':')]:
        val = limp['last_improvement_gen'].quantile(p / 100)
        ax.axvline(val, color=color, linestyle=ls, linewidth=1.8,
                   label=f'p{p} = gen {val:.0f}')
    ax.set_xlabel('Posledná generácia so zlepšením')
    ax.set_ylabel('Počet obrazkov')
    ax.set_title('Kedy nastalo posledné zlepšenie?')
    ax.legend(fontsize=9)

    # CDF — koľko % obrazkov by pokrylo patience=X
    ax = axes[1]
    sorted_vals = limp['last_improvement_gen'].sort_values().values
    cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals) * 100
    ax.plot(sorted_vals, cdf, color='steelblue', linewidth=2)
    for patience_val in [5, 10, 15, 25]:
        coverage = (limp['last_improvement_gen'] <= patience_val).mean() * 100
        ax.axvline(patience_val, color='gray', linestyle=':', linewidth=1)
        ax.text(patience_val + 0.3, coverage - 5,
                f'p={patience_val}\n{coverage:.0f}%',
                fontsize=8, color='#444444')
    ax.set_xlabel('patience hodnota')
    ax.set_ylabel('% pokrytých obrazkov')
    ax.set_title('Aké patience stačí pre X% obrazkov?')
    ax.set_ylim(0, 105)

    plt.suptitle('Analýza patience z existujúcich GA behov', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig('fig_patience_analysis.png', bbox_inches='tight')
    plt.show()

### EXP-2 – Porovnanie inicializačných metód GA

In [ ]:
INIT_METHODS = ['dm', 'gdm', 'random', 'quadtree', 'largest_rect']
INIT_ALGO_MAP = {
    'dm': 'ga_dm', 'gdm': 'ga_gdm', 'random': 'ga_random',
    'quadtree': 'ga_qtd', 'largest_rect': 'ga_lrf',
}
ANALYSIS_DATASETS = ['analysis/objects_unique', 'analysis/leafs_subset']

init_records = []
for method in INIT_METHODS:
    algo = INIT_ALGO_MAP[method]
    for ds in ANALYSIS_DATASETS:
        df = load_results(algo, ds)
        if df.empty:
            continue
        exp2 = df[df['run_path'].str.contains('exp2_init', na=False)]
        if exp2.empty:
            continue
        for _, row in exp2.iterrows():
            init_records.append({
                'init_method': method,
                'dataset': ds.split('/')[-1],
                'image_name': row['image_name'],
                'seed': row.get('seed'),
                'rectangle_count': row['rectangle_count'],
                'execution_time_sec': row['execution_time_sec'],
                'generations_used': row.get('generations_used'),
            })

init_df = pd.DataFrame(init_records)
if not init_df.empty:
    summary = (init_df.groupby(['init_method', 'dataset'])
               ['rectangle_count'].agg(['mean', 'std', 'count'])
               .round(2))
    display(summary)
else:
    print('EXP-2 results not available yet.')
    print('Run: python -m experiments.scripts.analysis.run_ga_exp2_init')

In [ ]:
if not init_df.empty:
    for ds in init_df['dataset'].unique():
        sub = init_df[init_df['dataset'] == ds]
        order = (sub.groupby('init_method')['rectangle_count']
                 .mean().sort_values().index.tolist())
        data_by_method = [sub[sub['init_method'] == m]['rectangle_count'].values
                          for m in order]

        fig, ax = plt.subplots(figsize=(7, 5))
        bp = ax.boxplot(data_by_method, labels=order, patch_artist=True,
                        medianprops={'color': 'black', 'linewidth': 2})
        for patch, color in zip(bp['boxes'], COLORS):
            patch.set_facecolor(color)
            patch.set_alpha(0.8)
        ax.set_ylabel('Počet obdĺžnikov')
        # ax.set_title(f'EXP-1: Porovnanie inicializacnych metód GA\n{ds.replace("_", " ")}')
        ax.tick_params(axis='x', rotation=30)
        plt.tight_layout()
        plt.savefig(f'fig_04_ga_init_{ds}.png', bbox_inches='tight')
        plt.show()

### EXP-3 – Porovnanie metód crossoveru GA

In [ ]:
CROSSOVER_METHODS = [
    'subset_greedy', 'subset_greedy_relaxed',
    'single_point', 'two_point', 'uniform',
]
BEST_INIT_ALGO = 'ga_dm'  # DM init — väčší priestor pre crossover ako GDM

cross_records = []
for ds in ANALYSIS_DATASETS:
    df = load_results(BEST_INIT_ALGO, ds)
    if df.empty:
        continue
    exp3 = df[df['run_path'].str.contains('exp3_crossover', na=False)]
    if exp3.empty:
        continue
    for _, row in exp3.iterrows():
        method = row['run_path'].split('exp3_crossover/')[-1].split('/')[0]
        cross_records.append({
            'crossover': method,
            'dataset': ds.split('/')[-1],
            'rectangle_count': row['rectangle_count'],
            'execution_time_sec': row['execution_time_sec'],
            'generations_used': row.get('generations_used'),
        })

cross_df = pd.DataFrame(cross_records)
if not cross_df.empty:
    summary = (cross_df.groupby(['crossover', 'dataset'])
               .agg(
                   mean_rects=('rectangle_count', 'mean'),
                   std_rects=('rectangle_count', 'std'),
                   mean_time=('execution_time_sec', 'mean'),
                   std_time=('execution_time_sec', 'std'),
                   n=('rectangle_count', 'count'),
               ).round(2))
    display(summary)
else:
    print('EXP-3 results not available yet.')
    print('Run: python -m experiments.scripts.analysis.run_ga_exp3_crossover')

In [ ]:
if not cross_df.empty:
    ds_list = cross_df['dataset'].unique()
    fig, axes = plt.subplots(1, len(ds_list), figsize=(8 * len(ds_list), 5))
    if len(ds_list) == 1:
        axes = [axes]
    for ax, ds in zip(axes, ds_list):
        sub = cross_df[cross_df['dataset'] == ds]
        order = (sub.groupby('crossover')['rectangle_count']
                 .mean().sort_values().index.tolist())
        data_by_method = [sub[sub['crossover'] == m]['rectangle_count'].values
                          for m in order]
        bp = ax.boxplot(data_by_method, labels=order, patch_artist=True,
                        medianprops={'color': 'black', 'linewidth': 2})
        for patch, color in zip(bp['boxes'], COLORS):
            patch.set_facecolor(color)
            patch.set_alpha(0.8)
        ax.set_ylabel('Rectangle count')
        ax.set_title(ds.replace('_', ' '))
        ax.tick_params(axis='x', rotation=30)
    plt.suptitle('EXP-3: GA Crossover Method Comparison', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig('fig_05_ga_crossover_comparison.png', bbox_inches='tight')
    plt.show()

### EXP-5 – Vplyv p_local a p_merge

In [ ]:
mut_records = []
for ds in DATASETS_COMPARE:
    df = load_results('ga_gdm', ds)
    if df.empty:
        continue
    exp5 = df[df['run_path'].str.contains('exp5_', na=False)].copy()
    if exp5.empty:
        continue
    exp5['exp'] = exp5['run_path'].str.extract(r'(exp5_\w+)')
    exp5['value'] = exp5['run_path'].str.extract(r'exp5_\w+/([\d.]+)').astype(float)
    for _, row in exp5.iterrows():
        mut_records.append({
            'exp': row['exp'],
            'value': row['value'],
            'dataset': ds,
            'image_name': row['image_name'],
            'rectangle_count': row['rectangle_count'],
            'execution_time_sec': row['execution_time_sec'],
        })

mut_df = pd.DataFrame(mut_records)
if not mut_df.empty:
    for exp_name in ['exp5_local', 'exp5_merge']:
        sub = mut_df[mut_df['exp'] == exp_name]
        if sub.empty:
            continue
        print(f"\n=== {exp_name} ===")
        print(sub.groupby(['value', 'dataset'])['rectangle_count']
              .agg(['mean', 'std', 'count']).round(2).to_string())
else:
    print('EXP-5 results not available yet.')
    print('Run: python -m experiments.scripts.run_ga_mutation_analysis')

In [ ]:
if not mut_df.empty:
    fig, axes = plt.subplots(2, len(DATASETS_COMPARE),
                             figsize=(6 * len(DATASETS_COMPARE), 9))
    if len(DATASETS_COMPARE) == 1:
        axes = axes.reshape(2, 1)

    labels = {
        'exp5_local': ('p_local', 'EXP-5A: Vplyv p_local (p_merge=0.10 fixné)'),
        'exp5_merge': ('p_merge', 'EXP-5B: Vplyv p_merge (p_local=0.50 fixné)'),
    }

    for row_i, (exp_name, (param, title)) in enumerate(labels.items()):
        for col_i, ds in enumerate(DATASETS_COMPARE):
            ax = axes[row_i][col_i]
            sub = mut_df[(mut_df['exp'] == exp_name) & (mut_df['dataset'] == ds)]
            if sub.empty:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                        transform=ax.transAxes)
                ax.set_title(f'{title}\n{ds}')
                continue

            stats = (sub.groupby('value')['rectangle_count']
                     .agg(['mean', 'std']).reset_index())
            ax.plot(stats['value'], stats['mean'], 'o-',
                    color='steelblue', linewidth=2, markersize=7)
            ax.fill_between(
                stats['value'],
                stats['mean'] - stats['std'],
                stats['mean'] + stats['std'],
                alpha=0.15, color='steelblue',
            )

            gdm_base = load_baseline('gdm', ds)
            gbd_base = load_baseline('graph_based', ds)
            if not gdm_base.empty:
                ax.axhline(gdm_base['rectangle_count'].mean(), color='orange',
                           linestyle='--', linewidth=1.5, label='GDM')
            if not gbd_base.empty:
                ax.axhline(gbd_base['rectangle_count'].mean(), color='green',
                           linestyle='--', linewidth=1.5, label='GBD')

            ax.set_xlabel(param)
            ax.set_ylabel('Mean rectangle count')
            ax.set_title(f'{ds.replace("_", " ")}')
            ax.legend(fontsize=9)

        # Row title
        fig.text(0.02, 0.75 - row_i * 0.5, title,
                 va='center', rotation='vertical', fontsize=11, color='#222222')

    plt.tight_layout(rect=[0.04, 0, 1, 1])
    plt.savefig('fig_11_mutation_analysis.png', bbox_inches='tight')
    plt.show()

### Konvergečná analýza (rect_count + fitness per generácia)

In [ ]:
CONV_INIT_METHODS = ['dm', 'gdm', 'random', 'quadtree', 'largest_rect']
CONV_ALGO_MAP = {
    'dm': 'ga_dm', 'gdm': 'ga_gdm', 'random': 'ga_random',
    'quadtree': 'ga_qtd', 'largest_rect': 'ga_lrf',
}

def load_conv_generations(dataset: str) -> pd.DataFrame:
    """Load conv_analysis generation histories for all init methods."""
    dfs = []
    for method, algo in CONV_ALGO_MAP.items():
        base = CSV_DIR / algo
        pattern = f"{dataset}/conv_analysis/{method}/*/generations.csv"
        for csv_path in sorted(base.glob(pattern)):
            try:
                df = pd.read_csv(csv_path)
                df['init_method'] = method
                if 'fitness' not in df.columns:
                    df['fitness'] = float('nan')
                dfs.append(df)
            except Exception:
                pass
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def plot_dual_convergence(gen_df: pd.DataFrame, dataset: str, save_prefix: str):
    """Plot rect_count and fitness convergence per init method side by side."""
    methods = sorted(gen_df['init_method'].unique())
    colors = {m: COLORS[i % len(COLORS)] for i, m in enumerate(methods)}

    fig, (ax_rect, ax_fit) = plt.subplots(1, 2, figsize=(14, 5))

    for method in methods:
        sub = gen_df[gen_df['init_method'] == method]
        color = colors[method]

        for ax, col in [
            (ax_rect, 'best_rectangle_count'),
            (ax_fit,  'fitness'),
        ]:
            stats = (sub.groupby('generation')[col]
                     .agg(['mean', 'std']).reset_index())
            ax.plot(stats['generation'], stats['mean'],
                    label=method, color=color, linewidth=2)
            ax.fill_between(
                stats['generation'],
                stats['mean'] - stats['std'].fillna(0),
                stats['mean'] + stats['std'].fillna(0),
                alpha=0.12, color=color,
            )

    ax_rect.set_xlabel('Generácia')
    ax_rect.set_ylabel('Počet obdĺžnikov')
    ax_rect.set_title('Počet obdĺžnikov per generácia')
    ax_rect.legend(fontsize=9)

    ax_fit.set_xlabel('Generácia')
    ax_fit.set_ylabel('Fitness')
    ax_fit.set_title('Fitness per generácia')
    ax_fit.legend(fontsize=9)

    plt.suptitle(
        f'GA Konvergencia – {dataset.replace("_", " ")} '
        f'(priemer cez obrazky ± std)',
        fontsize=13, y=1.02,
    )
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_{dataset}.png', bbox_inches='tight')
    plt.show()


for ds in DATASETS_COMPARE:
    gen_df = load_conv_generations(ds)
    if gen_df.empty:
        print(f'[{ds}] Convergence data not available yet.')
        print('Run: python -m experiments.scripts.run_ga_convergence_analysis')
    else:
        print(f'[{ds}] {len(gen_df)} rows, '
              f'{gen_df["image_name"].nunique()} images, '
              f'{gen_df["init_method"].nunique()} methods')
        plot_dual_convergence(gen_df, ds, save_prefix='fig_12_convergence')

## 3. Full Run – Porovnanie algoritmov

Finálne porovnanie na kompletných datasetoch `leafs_selected` (204) a `objects_selected` (282).

## 3. Full Run – Porovnanie algoritmov

In [ ]:
records = []
for algo in DETERMINISTIC_ALGOS:
    for ds in DATASETS_COMPARE:
        df = load_baseline(algo, ds)
        if df.empty:
            continue
        records.append({
            'algorithm': algo, 'dataset': ds, 'n': len(df),
            'mean_rect': df['rectangle_count'].mean(),
            'std_rect': df['rectangle_count'].std(),
            'median_rect': df['rectangle_count'].median(),
            'mean_time': df['execution_time_sec'].mean(),
        })

for algo in GA_ALGOS:
    for ds in DATASETS_COMPARE:
        df = load_ga_baseline(algo, ds)
        if df.empty:
            continue
        records.append({
            'algorithm': algo, 'dataset': ds, 'n': len(df),
            'mean_rect': df['rectangle_count'].mean(),
            'std_rect': df['rectangle_count'].std(),
            'median_rect': df['rectangle_count'].median(),
            'mean_time': df['execution_time_sec'].mean(),
        })

baseline_df = pd.DataFrame(records)
if not baseline_df.empty:
    display(baseline_df.round(1))
else:
    print('No baseline results available yet.')

In [ ]:
PLOT_ALGOS = [
    'dm',
    'gdm',
    # 'quadtree',
    'largest_rect',
    'graph_based',
]

if not baseline_df.empty:
    COLORS = plt.cm.Set2.colors

    det_df = baseline_df[baseline_df['algorithm'].isin(PLOT_ALGOS)]

    datasets_available = det_df['dataset'].unique()
    fig, axes = plt.subplots(
        1, len(datasets_available),
        figsize=(7 * len(datasets_available), 5),
        sharey=False,
    )
    if len(datasets_available) == 1:
        axes = [axes]

    for ax, ds in zip(axes, datasets_available):
        sub = det_df[det_df['dataset'] == ds].sort_values('mean_rect')
        colors = [COLORS[i % len(COLORS)] for i in range(len(sub))]
        bars = ax.barh(
            sub['algorithm'], sub['mean_rect'],
            xerr=sub['std_rect'], color=colors,
            capsize=3, edgecolor='white', alpha=0.85,
        )
        ax.set_xlabel('Mean rectangle count')
        ax.set_title(ds.replace('_', ' '))
        for bar, (_, row) in zip(bars, sub.iterrows()):
            label = f"{row['mean_rect']:.1f}  (n={int(row['n'])})"
            ax.text(
                row['mean_rect'] + row['std_rect'] + 0.5,
                bar.get_y() + bar.get_height() / 2,
                label, va='center', fontsize=8,
            )

    plt.suptitle(
        'Algorithm Comparison – Mean Rectangle Count (deterministic, full dataset)',
        fontsize=13, y=1.02,
    )
    plt.tight_layout()
    plt.savefig('fig_02_algorithm_comparison.png', bbox_inches='tight')
    plt.show()

In [ ]:
if not baseline_df.empty:
    ds_list = baseline_df['dataset'].unique()
    fig, axes = plt.subplots(1, len(ds_list), figsize=(6 * len(ds_list), 4))
    if len(ds_list) == 1:
        axes = [axes]
    for ax, ds in zip(axes, ds_list):
        sub = baseline_df[
            (baseline_df['dataset'] == ds)
            & (baseline_df['algorithm'] != 'quadtree')
        ]
        for i, (_, row) in enumerate(sub.iterrows()):
            ax.scatter(row['mean_time'], row['mean_rect'],
                       s=120, color=COLORS[i % len(COLORS)], zorder=3)
            ax.annotate(row['algorithm'], (row['mean_time'], row['mean_rect']),
                        textcoords='offset points', xytext=(5, 3), fontsize=9)
        ax.set_xlabel('Mean execution time (s)')
        ax.set_ylabel('Mean rectangle count')
        ax.set_title(ds.replace('_', ' '))
    plt.suptitle('Time vs. Quality Trade-off', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig('fig_03_time_vs_quality.png', bbox_inches='tight')
    plt.show()

## Súhrnná tabuľka

### GA vs GDM vs GBD – Skóre kvality a časová efektivita

In [ ]:
for ds in DATASETS_COMPARE:
    gdm_df = load_baseline('gdm', ds)
    ga_df  = load_ga_baseline('ga_gdm', ds)
    if gdm_df.empty or ga_df.empty:
        print(f'[{ds}] Chýbajú dáta.')
        continue

    merged = gdm_df[['image_name', 'rectangle_count']].merge(
        ga_df[['image_name', 'rectangle_count', 'execution_time_sec']],
        on='image_name', suffixes=('_gdm', '_ga'),
    )
    if merged.empty:
        print(f'[{ds}] Žiadne spoločné obrázky.')
        continue

    merged['diff']        = merged['rectangle_count_ga'] - merged['rectangle_count_gdm']
    merged['improvement'] = -merged['diff']
    merged['improved']    = merged['diff'] < 0
    merged['pct_improve'] = merged['improvement'] / merged['rectangle_count_gdm'] * 100

    merged['complexity_bin'] = pd.qcut(
        merged['rectangle_count_gdm'], q=4,
        labels=['Q1\n(jednoduchý)', 'Q2', 'Q3', 'Q4\n(zložitý)']
    )

    print(f'\n=== {ds} — {len(merged)} spoločných obrázkov ===')
    stats = merged.groupby('complexity_bin', observed=True).agg(
        n=('image_name', 'count'),
        gdm_mean=('rectangle_count_gdm', 'mean'),
        ga_mean=('rectangle_count_ga', 'mean'),
        pct_improved=('improved', lambda x: f"{x.mean()*100:.0f}%"),
        mean_improvement=('improvement', 'mean'),
        mean_pct_improve=('pct_improve', 'mean'),
        mean_time_ga=('execution_time_sec', 'mean'),
    ).round(1)
    display(stats)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    colors_scatter = merged['improvement'].apply(
        lambda v: 'steelblue' if v > 0 else ('tomato' if v < 0 else 'lightgray')
    )
    ax.scatter(merged['rectangle_count_gdm'], merged['improvement'],
               c=colors_scatter, alpha=0.7, s=50, edgecolors='none')
    ax.axhline(0, color='#888', linewidth=1, linestyle='--')
    ax.set_xlabel('GDM rect count (zložitosť obrázka)')
    ax.set_ylabel('Zlepšenie GA oproti GDM (rects)')
    ax.set_title('Zlepšenie GA vs. zložitosť obrázka')
    ax.legend(handles=[
        mpatches.Patch(color='steelblue', label='GA lepší'),
        mpatches.Patch(color='lightgray',  label='rovnaké'),
        mpatches.Patch(color='tomato',     label='GA horší'),
    ], fontsize=9)

    ax = axes[1]
    bin_stats = merged.groupby('complexity_bin', observed=True).agg(
        pct_improved=('improved', 'mean'),
        mean_pct=('pct_improve', 'mean'),
    )
    x = range(len(bin_stats))
    ax.bar(x, bin_stats['pct_improved'] * 100,
           color='steelblue', alpha=0.8, edgecolor='white')
    ax2 = ax.twinx()
    ax2.plot(x, bin_stats['mean_pct'], 'o--',
             color='tomato', linewidth=2, markersize=7, label='priem. % zlepšenie')
    ax.set_xticks(list(x))
    ax.set_xticklabels(bin_stats.index.tolist())
    ax.set_ylabel('% obrázkov kde GA vyhrá')
    ax2.set_ylabel('Priemerné zlepšenie (%)', color='tomato')
    ax2.tick_params(axis='y', labelcolor='tomato')
    ax.set_title('GA win rate a zlepšenie podľa kvartilu')
    ax2.legend(loc='upper left', fontsize=9)

    plt.suptitle(f'GA vs GDM – stratifikovaná analýza ({ds.replace("_", " ")})',
                 fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(f'fig_ga_vs_gdm_stratified_{ds}.png', bbox_inches='tight')
    plt.show()

In [ ]:
for ds in DATASETS_COMPARE:
    gdm_raw = load_baseline('gdm', ds)
    gbd_raw = load_baseline('graph_based', ds)
    ga_raw  = load_ga_baseline('ga_gdm', ds)

    if gdm_raw.empty or gbd_raw.empty or ga_raw.empty:
        print(f'[{ds}] Chýbajú dáta (gdm={len(gdm_raw)}, gbd={len(gbd_raw)}, ga={len(ga_raw)}).')
        continue

    gdm_df = gdm_raw[['image_name', 'rectangle_count', 'execution_time_sec']].rename(
        columns={'rectangle_count': 'rects_gdm', 'execution_time_sec': 'time_gdm'})
    gbd_df = gbd_raw[['image_name', 'rectangle_count', 'execution_time_sec']].rename(
        columns={'rectangle_count': 'rects_gbd', 'execution_time_sec': 'time_gbd'})
    ga_df  = ga_raw[['image_name', 'rectangle_count', 'execution_time_sec']].rename(
        columns={'rectangle_count': 'rects_ga', 'execution_time_sec': 'time_ga'})

    m = gdm_df.merge(gbd_df, on='image_name').merge(ga_df, on='image_name')
    if m.empty:
        print(f'[{ds}] Žiadne spoločné obrázky.')
        continue

    # Metriky
    m['gap_from_opt_pct']     = (m['rects_ga']  - m['rects_gbd']) / m['rects_gbd'] * 100
    m['improvement_gdm_pct']  = (m['rects_gdm'] - m['rects_ga'])  / m['rects_gdm'] * 100
    m['gdm_gap_from_opt_pct'] = (m['rects_gdm'] - m['rects_gbd']) / m['rects_gbd'] * 100
    # TOS: (GBD_time / GA_time) / (GA_rects / GBD_rects) — >1: GA sa oplatí
    m['TOS'] = (m['time_gbd'] / m['time_ga'].clip(lower=0.01)) / (m['rects_ga'] / m['rects_gbd'].clip(lower=1))

    m['complexity_bin'] = pd.qcut(
        m['rects_gdm'], q=4,
        labels=['Q1\n(jednoduchý)', 'Q2', 'Q3', 'Q4\n(zložitý)']
    )

    print(f'\n=== {ds} — {len(m)} obrázkov ===')
    summary = m.groupby('complexity_bin', observed=True).agg(
        n=('image_name', 'count'),
        gdm_mean=('rects_gdm', 'mean'),
        ga_mean=('rects_ga', 'mean'),
        gbd_mean=('rects_gbd', 'mean'),
        gap_from_opt=('gap_from_opt_pct', 'mean'),
        gdm_gap_from_opt=('gdm_gap_from_opt_pct', 'mean'),
        improvement_gdm=('improvement_gdm_pct', 'mean'),
        TOS_mean=('TOS', 'mean'),
        TOS_pct_above1=('TOS', lambda x: f"{(x > 1).mean()*100:.0f}%"),
    ).round(2)
    display(summary)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    ax = axes[0]
    ax.scatter(m['rects_gbd'], m['gap_from_opt_pct'],
               color='steelblue', alpha=0.6, s=40, label='GA', edgecolors='none')
    ax.scatter(m['rects_gbd'], m['gdm_gap_from_opt_pct'],
               color='orange', alpha=0.5, s=40, label='GDM', edgecolors='none', marker='s')
    ax.axhline(0, color='green', linewidth=1.2, linestyle='--', label='Optimum (GBD)')
    ax.set_xlabel('GBD rect count (optimum)')
    ax.set_ylabel('% nad optimom')
    ax.set_title('Vzdialenosť od optima')
    ax.legend(fontsize=9)

    ax = axes[1]
    bin_stats = m.groupby('complexity_bin', observed=True)['improvement_gdm_pct'].agg(['mean', 'std'])
    x = range(len(bin_stats))
    ax.bar(x, bin_stats['mean'], yerr=bin_stats['std'],
           color='steelblue', alpha=0.8, edgecolor='white', capsize=4)
    ax.axhline(0, color='#888', linewidth=1, linestyle='--')
    ax.set_xticks(list(x))
    ax.set_xticklabels(bin_stats.index.tolist())
    ax.set_ylabel('Zlepšenie GA vs GDM (%)')
    ax.set_title('GA zlepšenie podľa zložitosti')

    ax = axes[2]
    tos_stats = m.groupby('complexity_bin', observed=True)['TOS'].agg(['mean', 'std'])
    colors_tos = ['steelblue' if v >= 1 else 'tomato' for v in tos_stats['mean']]
    ax.bar(x, tos_stats['mean'], yerr=tos_stats['std'],
           color=colors_tos, alpha=0.8, edgecolor='white', capsize=4)
    ax.axhline(1, color='green', linewidth=1.5, linestyle='--', label='TOS=1 (break-even)')
    ax.set_xticks(list(x))
    ax.set_xticklabels(tos_stats.index.tolist())
    ax.set_ylabel('TOS  (GBD_time/GA_time) / (GA_rects/GBD_rects)')
    ax.set_title('Časová efektivita GA vs GBD')
    ax.legend(fontsize=9)

    plt.suptitle(f'GA vs GDM vs GBD – skóre ({ds.replace("_", " ")})', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(f'fig_ga_scores_{ds}.png', bbox_inches='tight')
    plt.show()

In [ ]:
# Combined summary: best GA config vs. all deterministic methods
summary_records = []
for algo in DETERMINISTIC_ALGOS:
    for ds in DATASETS_COMPARE:
        df = load_baseline(algo, ds)
        if df.empty:
            continue
        summary_records.append({
            'algorithm': algo,
            'dataset': ds.replace('_', ' '),
            'n': len(df),
            'mean ± std': f"{df['rectangle_count'].mean():.1f} ± {df['rectangle_count'].std():.1f}",
            'median': df['rectangle_count'].median(),
            'min': df['rectangle_count'].min(),
            'max': df['rectangle_count'].max(),
            'mean_time_s': df['execution_time_sec'].mean().round(1),
        })

# Add best GA run if available
for ds in DATASETS_COMPARE:
    df = load_results('ga_gdm', ds)
    if df.empty:
        continue
    df = df[~df['run_path'].str.contains('test|exp2|exp3', na=False)]
    if df.empty:
        continue
    summary_records.append({
        'algorithm': 'ga_gdm (best)',
        'dataset': ds.replace('_', ' '),
        'n': len(df),
        'mean ± std': f"{df['rectangle_count'].mean():.1f} ± {df['rectangle_count'].std():.1f}",
        'median': df['rectangle_count'].median(),
        'min': df['rectangle_count'].min(),
        'max': df['rectangle_count'].max(),
        'mean_time_s': df['execution_time_sec'].mean().round(1),
    })

if summary_records:
    summary_df = pd.DataFrame(summary_records)
    for ds in summary_df['dataset'].unique():
        print(f'\n=== {ds} ===')
        sub = summary_df[summary_df['dataset'] == ds].drop(columns='dataset')
        print(sub.to_string(index=False))
else:
    print('No results available for summary. Run experiments first.')